# LeetCode #460: LFU Cache

https://leetcode.com/problems/lfu-cache/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Naive (Linear Scan for Min Freq)** | get $O(1)$, put $O(n)$ | $O(n)$ |
| **Optimal: Two HashMaps + Doubly Linked Lists ★** | get $O(1)$, put $O(1)$ | $O(n)$ |

---

## Understanding the Methods

### Naive (Linear Scan for Min Freq)
Store all key-value pairs with their frequencies. On eviction, scan all entries to find the one with the lowest frequency (and oldest access if tied). Get is O(1) via a hash map, but put requires O(n) for eviction.

### Optimal: Two HashMaps + Doubly Linked Lists ★
Use a key-to-node hash map for O(1) lookup, and a frequency-to-doubly-linked-list hash map where each list holds nodes at that frequency (ordered by recency). Track the minimum frequency. On get/put, move the node to frequency+1 list. On eviction, remove the tail (LRU) of the min-frequency list.

**Why this is better than Naive:** By maintaining a min-frequency pointer and per-frequency LRU lists, both get and put run in O(1) amortized time.

**Constraints:**
* 1 <= capacity <= 10^4
* 0 <= key <= 10^5
* 0 <= value <= 10^9
* At most 2 * 10^5 calls to get and put

## Solutions

### C#

In [ ]:
public class LFUCache {
    private int capacity, minFreq;
    private Dictionary<int, (int val, int freq)> keyMap;
    private Dictionary<int, LinkedList<int>> freqMap;
    private Dictionary<int, LinkedListNode<int>> nodeMap;

    public LFUCache(int capacity) {
        this.capacity = capacity;
        minFreq = 0;
        keyMap = new Dictionary<int, (int, int)>();
        freqMap = new Dictionary<int, LinkedList<int>>();
        nodeMap = new Dictionary<int, LinkedListNode<int>>();
    }

    public int Get(int key) {
        if (!keyMap.ContainsKey(key)) return -1;
        var (val, freq) = keyMap[key];
        Touch(key, freq);
        keyMap[key] = (val, freq + 1);
        return val;
    }

    public void Put(int key, int value) {
        if (capacity == 0) return;
        if (keyMap.ContainsKey(key)) {
            var (_, freq) = keyMap[key];
            Touch(key, freq);
            keyMap[key] = (value, freq + 1);
        } else {
            if (keyMap.Count >= capacity) Evict();
            keyMap[key] = (value, 1);
            if (!freqMap.ContainsKey(1)) freqMap[1] = new LinkedList<int>();
            freqMap[1].AddFirst(key);
            nodeMap[key] = freqMap[1].First;
            minFreq = 1;
        }
    }

    private void Touch(int key, int freq) {
        freqMap[freq].Remove(nodeMap[key]);
        if (freqMap[freq].Count == 0) {
            freqMap.Remove(freq);
            if (minFreq == freq) minFreq++;
        }
        int nf = freq + 1;
        if (!freqMap.ContainsKey(nf)) freqMap[nf] = new LinkedList<int>();
        freqMap[nf].AddFirst(key);
        nodeMap[key] = freqMap[nf].First;
    }

    private void Evict() {
        var list = freqMap[minFreq];
        int evictKey = list.Last.Value;
        list.RemoveLast();
        if (list.Count == 0) freqMap.Remove(minFreq);
        keyMap.Remove(evictKey);
        nodeMap.Remove(evictKey);
    }
}

### Python

In [ ]:
from collections import defaultdict, OrderedDict

class LFUCache:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.min_freq = 0
        self.key_map = {}  # key -> (val, freq)
        self.freq_map = defaultdict(OrderedDict)  # freq -> OrderedDict of key->None

    def get(self, key: int) -> int:
        if key not in self.key_map:
            return -1
        val, freq = self.key_map[key]
        self._touch(key, freq)
        self.key_map[key] = (val, freq + 1)
        return val

    def put(self, key: int, value: int) -> None:
        if self.cap == 0:
            return
        if key in self.key_map:
            _, freq = self.key_map[key]
            self._touch(key, freq)
            self.key_map[key] = (value, freq + 1)
        else:
            if len(self.key_map) >= self.cap:
                evict_key, _ = self.freq_map[self.min_freq].popitem(last=False)
                if not self.freq_map[self.min_freq]:
                    del self.freq_map[self.min_freq]
                del self.key_map[evict_key]
            self.key_map[key] = (value, 1)
            self.freq_map[1][key] = None
            self.min_freq = 1

    def _touch(self, key, freq):
        del self.freq_map[freq][key]
        if not self.freq_map[freq]:
            del self.freq_map[freq]
            if self.min_freq == freq:
                self.min_freq += 1
        self.freq_map[freq + 1][key] = None

### Go

In [ ]:
import "container/list"

type entry struct {
    key, val, freq int
}

type LFUCache struct {
    cap, minFreq int
    keyMap       map[int]*list.Element
    freqMap      map[int]*list.List
}

func Constructor(capacity int) LFUCache {
    return LFUCache{cap: capacity, keyMap: map[int]*list.Element{}, freqMap: map[int]*list.List{}}
}

func (c *LFUCache) Get(key int) int {
    el, ok := c.keyMap[key]
    if !ok { return -1 }
    e := el.Value.(*entry)
    c.touch(e)
    return e.val
}

func (c *LFUCache) Put(key int, value int) {
    if c.cap == 0 { return }
    if el, ok := c.keyMap[key]; ok {
        e := el.Value.(*entry)
        e.val = value
        c.touch(e)
        return
    }
    if len(c.keyMap) >= c.cap {
        lst := c.freqMap[c.minFreq]
        back := lst.Back()
        lst.Remove(back)
        if lst.Len() == 0 { delete(c.freqMap, c.minFreq) }
        delete(c.keyMap, back.Value.(*entry).key)
    }
    e := &entry{key, value, 1}
    if c.freqMap[1] == nil { c.freqMap[1] = list.New() }
    c.keyMap[key] = c.freqMap[1].PushFront(e)
    c.minFreq = 1
}

func (c *LFUCache) touch(e *entry) {
    lst := c.freqMap[e.freq]
    lst.Remove(c.keyMap[e.key])
    if lst.Len() == 0 {
        delete(c.freqMap, e.freq)
        if c.minFreq == e.freq { c.minFreq++ }
    }
    e.freq++
    if c.freqMap[e.freq] == nil { c.freqMap[e.freq] = list.New() }
    c.keyMap[e.key] = c.freqMap[e.freq].PushFront(e)
}

### Rust

In [ ]:
use std::collections::HashMap;

struct LFUCache {
    cap: usize,
    min_freq: usize,
    key_map: HashMap<i32, (i32, usize)>,       // key -> (val, freq)
    freq_map: HashMap<usize, Vec<i32>>,         // freq -> keys in order
    key_pos: HashMap<i32, usize>,               // key -> position in freq list
    tick: usize,
    tick_map: HashMap<i32, usize>,
}

// Note: A production-quality O(1) LFU in Rust requires an intrusive
// doubly-linked list (unsafe) or an IndexMap crate. This simplified
// version uses a BTreeMap<(freq, tick), key> for eviction ordering.
use std::collections::BTreeMap;

struct LFUCache {
    cap: usize,
    tick: usize,
    key_map: HashMap<i32, (i32, usize, usize)>, // key -> (val, freq, tick)
    order: BTreeMap<(usize, usize), i32>,        // (freq, tick) -> key
}

impl LFUCache {
    fn new(capacity: i32) -> Self {
        LFUCache {
            cap: capacity as usize, tick: 0,
            key_map: HashMap::new(), order: BTreeMap::new(),
        }
    }

    fn get(&mut self, key: i32) -> i32 {
        if let Some(&(val, freq, old_tick)) = self.key_map.get(&key) {
            self.order.remove(&(freq, old_tick));
            self.tick += 1;
            self.key_map.insert(key, (val, freq + 1, self.tick));
            self.order.insert((freq + 1, self.tick), key);
            val
        } else { -1 }
    }

    fn put(&mut self, key: i32, value: i32) {
        if self.cap == 0 { return; }
        if self.key_map.contains_key(&key) {
            let (_, freq, old_tick) = self.key_map[&key];
            self.order.remove(&(freq, old_tick));
            self.tick += 1;
            self.key_map.insert(key, (value, freq + 1, self.tick));
            self.order.insert((freq + 1, self.tick), key);
        } else {
            if self.key_map.len() >= self.cap {
                let (&k, &evict_key) = self.order.iter().next().unwrap();
                self.order.remove(&k);
                self.key_map.remove(&evict_key);
            }
            self.tick += 1;
            self.key_map.insert(key, (value, 1, self.tick));
            self.order.insert((1, self.tick), key);
        }
    }
}

## Example Scenarios

### Scenario 1: Basic LFU eviction
**Input:** capacity=2, put(1,1), put(2,2), get(1), put(3,3), get(2), get(3)  
After get(1), key 1 has freq 2, key 2 has freq 1. put(3,3) evicts key 2 (lowest freq). get(2) returns -1. **Output:** `[1,-1,3]`

### Scenario 2: Tie-breaking by recency
**Input:** capacity=2, put(1,1), put(2,2), put(3,3)  
Both keys 1 and 2 have freq 1. Key 1 was used least recently, so it is evicted. **Output:** cache contains {2:2, 3:3}

### Scenario 3: Update existing key
**Input:** capacity=2, put(1,1), put(1,10), get(1)  
put(1,10) updates value and increments frequency. get(1) returns 10. **Output:** `[10]`

### Scenario 4: Capacity 1
**Input:** capacity=1, put(1,1), put(2,2), get(1), get(2)  
put(2,2) evicts key 1. get(1) returns -1, get(2) returns 2. **Output:** `[-1,2]`

### Scenario 5: Zero capacity
**Input:** capacity=0, put(1,1), get(1)  
Nothing can be stored. get always returns -1. **Output:** `[-1]`

![image](attachment:image.png)